In [1]:
import numpy as np

np.random.seed(42)
X = np.random.uniform(-2,2,(400,3))
y = (np.sin(X[:,0]) + 0.5*(X[:,1]**2) - 0.8*X[:,2]).reshape(-1,1)

X = X.T
y = y.T

In [2]:
def relu(z):
    return np.maximum(0,z)

In [3]:
def drelu(z):
    return (z>0).astype(float)

In [4]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [5]:
def dsigmoid(z):
    s = sigmoid(z)
    return s*(1-s)

In [6]:
def tanh(z):
    return np.tanh(z)

In [7]:
def dtanh(z):
    return 1-np.tanh(z)**2

In [8]:
def leaky_relu(z):
    return np.where(z>0,z,0.01*z)

In [9]:
def dleaky_relu(z):
    return np.where(z>0,1,0.01)

In [10]:
def softplus(z):
    return np.log(1+np.exp(z))

In [11]:
def dsoftplus(z):
    return sigmoid(z)

In [12]:
def init_params(layers):
    params={}
    for l in range(1,len(layers)):
        params["W"+str(l)] = np.random.uniform(-0.5,0.5,(layers[l],layers[l-1]))
        params["b"+str(l)] = np.zeros((layers[l],1))
    return params

In [13]:
def forward(X,params,activation):
    cache={"A0":X}
    L=len(params)//2
    for l in range(1,L):
        Z=params["W"+str(l)]@cache["A"+str(l-1)]+params["b"+str(l)]
        cache["Z"+str(l)]=Z
        cache["A"+str(l)]=activation(Z)
    Z=params["W"+str(L)]@cache["A"+str(L-1)]+params["b"+str(L)]
    cache["Z"+str(L)]=Z
    cache["A"+str(L)]=Z
    return cache

In [14]:
def backward(params,cache,y,activation_deriv):
    grads={}
    m=y.shape[1]
    L=len(params)//2
    dA=2*(cache["A"+str(L)]-y)
    for l in reversed(range(1,L+1)):
        if l==L:
            dZ=dA
        else:
            dZ=dA*activation_deriv(cache["Z"+str(l)])
        grads["dW"+str(l)]=(1/m)*(dZ@cache["A"+str(l-1)].T)
        grads["db"+str(l)]=(1/m)*np.sum(dZ,axis=1,keepdims=True)
        dA=params["W"+str(l)].T@dZ
    return grads

In [15]:
def update(params,grads,lr):
    L=len(params)//2
    for l in range(1,L+1):
        params["W"+str(l)]-=lr*grads["dW"+str(l)]
        params["b"+str(l)]-=lr*grads["db"+str(l)]
    return params

In [16]:
def train(layers,activation,activation_deriv):
    params=init_params(layers)
    lr=0.01
    epochs=1000
    loss200=0
    for i in range(epochs):
        cache=forward(X,params,activation)
        loss=np.mean((y-cache["A"+str(len(layers)-1)])**2)
        grads=backward(params,cache,y,activation_deriv)
        params=update(params,grads,lr)
        if i==199:
            loss200=loss
    g1=np.linalg.norm(grads["dW1"])
    glast=np.linalg.norm(grads["dW"+str(len(layers)-1)])
    return loss,loss200,g1,glast

In [17]:
models={
"A":[3,4,1],
"B":[3,6,6,1],
"C":[3,8,8,8,8,1],
"D":[3,8,8,8,8,8,8,8,8,1]
}

for name,layers in models.items():
    loss,loss200,g1,glast=train(layers,relu,drelu)
    print(name,"ReLU",loss,loss200,g1,glast)

for name,layers in models.items():
    loss,loss200,g1,glast=train(layers,sigmoid,dsigmoid)
    print(name,"Sigmoid",loss,loss200,g1,glast)

A ReLU 0.11154512827725771 0.4938471859377117 0.045217360748741345 0.037272667469061424
B ReLU 0.07288586596130892 0.32198656685903304 0.03660874595282526 0.017981745565465418
C ReLU 0.030451498206553077 0.8620099934569575 0.02387634547844159 0.008826024106400016
D ReLU 0.0531848271858129 1.6348671624715951 0.42978396573665373 0.43046150169752845
A Sigmoid 0.4034798512536561 1.1076094569139452 0.02328218387802834 0.045953322221203505
B Sigmoid 0.6847471250676317 1.6871598726753871 0.20507396604773612 0.2656096276147131
C Sigmoid 1.7431256770560626 1.743379498003159 0.001908655651767695 0.002253691487489325
D Sigmoid 1.743852977779307 1.7438529782248355 2.517545785570898e-06 1.4009656550804531e-06
